# Prithvi WxC MERRA-2 Chemistry Fine-Tuning


This notebook runs the MERRA-2 next-step chemistry training pipeline using the scripts in `examples/merra2_prediction/`.

Workflow:
1. Configure paths/species/predictors in YAML (or notebook overrides).
2. Optionally run preprocessing and scaler computation.
3. Fine-tune checkpoint.
4. Inspect saved checkpoints and validation metrics.


In [ ]:
from __future__ import annotations

import os
import sys
import json
import subprocess
from datetime import datetime
from pathlib import Path

# -----------------------------
# USER CONFIGURATION (EDIT ME)
# -----------------------------
CONFIG_PATH = Path("examples/merra2_prediction/o3_pipeline_config.yaml")

# Optional runtime overrides merged into YAML before running scripts.
# Example target swap:
# {
#   "data": {
#      "target_var": "CO",
#      "target_input_name": "CO_sfc",
#      "target_transform": "surface",
#      "target_name": "CO_sfc_target",
#      "chem_predictors": [{"var": "O3", "output_name": "O3_sfc", "transform": "surface"}],
#      "predictor_vars": ["CO_sfc", "O3_sfc", "T_sfc", "U_sfc", "V_sfc", "PS_sfc"],
#   }
# }
OVERRIDES = {}

RUN_PREPROCESS = False
OVERWRITE_PREPROCESS = False
RUN_SCALERS = False
RUN_FINETUNE = True

RUN_NAME = f"chem_ft_{datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}"
PYTHON_BIN = os.environ.get("PYTHON_BIN", sys.executable)

# Optional explicit base override (directory containing merra2/, checkpoints/, outputs/ ...)
# os.environ["MERRA2_PREDICTION_BASE"] = "/abs/path/to/examples/merra2_prediction"


In [ ]:
import torch

print("Python:", sys.version.split()[0])
print("Python executable:", PYTHON_BIN)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"  {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
import yaml


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "examples" / "merra2_prediction").exists():
            return p
    raise FileNotFoundError("Could not locate repo root containing examples/merra2_prediction")


def deep_update(dst: dict, src: dict) -> dict:
    for k, v in src.items():
        if isinstance(v, dict) and isinstance(dst.get(k), dict):
            deep_update(dst[k], v)
        else:
            dst[k] = v
    return dst


REPO_ROOT = find_repo_root(Path.cwd())
if not CONFIG_PATH.is_absolute():
    CONFIG_PATH = (REPO_ROOT / CONFIG_PATH).resolve()

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config not found: {CONFIG_PATH}")

cfg = yaml.safe_load(CONFIG_PATH.read_text())
if not isinstance(cfg, dict):
    raise TypeError("YAML config must parse to a dict")

cfg_runtime = json.loads(json.dumps(cfg))
if OVERRIDES:
    deep_update(cfg_runtime, OVERRIDES)

runtime_cfg_path = CONFIG_PATH.with_suffix(".runtime.yaml")
runtime_cfg_path.write_text(yaml.safe_dump(cfg_runtime, sort_keys=False), encoding="utf-8")

print("Repo root:", REPO_ROOT)
print("Base config:", CONFIG_PATH)
print("Runtime config:", runtime_cfg_path)
print("Applied overrides:", bool(OVERRIDES))


In [ ]:
data_cfg = cfg_runtime.get("data", {})
train_cfg = cfg_runtime.get("training", {})
path_cfg = cfg_runtime.get("paths", {})

print("=== Data Setup ===")
print("target_var       :", data_cfg.get("target_var", data_cfg.get("chem_var", "O3")))
print("target_input_name:", data_cfg.get("target_input_name", data_cfg.get("o3_output_name", "O3_sfc")))
print("target_transform :", data_cfg.get("target_transform", "surface"))
print("target_name      :", data_cfg.get("target_name", "<auto>"))
print("delta_hours      :", data_cfg.get("delta_hours", 3))
print("chem_predictors  :", data_cfg.get("chem_predictors", []))
print("met_vars         :", data_cfg.get("met_vars", ["T", "U", "V", "PS"]))
print("predictor_vars   :", data_cfg.get("predictor_vars", None))

print("
=== Training Setup ===")
print("run_name         :", RUN_NAME)
print("epochs           :", train_cfg.get("num_epochs", 10))
print("batch_size       :", train_cfg.get("batch_size", 1))
print("learning_rate    :", train_cfg.get("learning_rate", 1.0e-4))
print("device           :", train_cfg.get("device", "auto"))
print("use_pretrained   :", train_cfg.get("use_pretrained_backbone", True))
print("init_checkpoint  :", train_cfg.get("initialize_from_checkpoint", None))

print("
=== Path Hints ===")
for key in ["chem_root", "met_root", "preprocessed_dir", "scalers_dir", "checkpoints_dir"]:
    print(f"{key:16s}:", path_cfg.get(key, "<auto>"))


In [ ]:
def run_cmd(cmd, cwd=None):
    print("+", " ".join(str(x) for x in cmd))
    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed with exit code {rc}: {' '.join(map(str, cmd))}")


def script_path(name: str) -> Path:
    return (REPO_ROOT / "examples" / "merra2_prediction" / name).resolve()


In [ ]:
if RUN_PREPROCESS:
    cmd = [
        PYTHON_BIN,
        script_path("preprocess_o3_pairs.py"),
        "--config",
        runtime_cfg_path,
    ]
    if OVERWRITE_PREPROCESS:
        cmd.append("--overwrite")
    run_cmd(cmd, cwd=REPO_ROOT)
else:
    print("RUN_PREPROCESS=False, skipping preprocess step")


In [ ]:
if RUN_SCALERS:
    cmd = [
        PYTHON_BIN,
        script_path("compute_scalars_o3.py"),
        "--config",
        runtime_cfg_path,
    ]
    run_cmd(cmd, cwd=REPO_ROOT)
else:
    print("RUN_SCALERS=False, skipping scaler step")


In [ ]:
if RUN_FINETUNE:
    cmd = [
        PYTHON_BIN,
        script_path("finetune_o3_next_step.py"),
        "--config",
        runtime_cfg_path,
        "--run-name",
        RUN_NAME,
    ]
    run_cmd(cmd, cwd=REPO_ROOT)
else:
    print("RUN_FINETUNE=False, skipping fine-tuning step")


In [ ]:
import glob

ckpt_root = (REPO_ROOT / "examples" / "merra2_prediction" / "checkpoints").resolve()
run_dir = ckpt_root / RUN_NAME
print("Run directory:", run_dir)

if run_dir.exists():
    files = sorted(glob.glob(str(run_dir / "*")))
    for f in files:
        print(Path(f).name)
else:
    print("Run directory does not exist yet.")


In [ ]:
import torch
import matplotlib.pyplot as plt

ckpt_root = (REPO_ROOT / "examples" / "merra2_prediction" / "checkpoints").resolve()
run_dir = ckpt_root / RUN_NAME
epoch_paths = sorted(run_dir.glob("epoch_*.ckpt"))

if not epoch_paths:
    print("No epoch checkpoints found for", RUN_NAME)
else:
    epochs, train_loss, val_loss, train_rmse, val_rmse = [], [], [], [], []
    for p in epoch_paths:
        d = torch.load(p, map_location="cpu", weights_only=False)
        epochs.append(int(d.get("epoch", len(epochs) + 1)))
        train_loss.append(float(d.get("train_loss", float("nan"))))
        val_loss.append(float(d.get("val_loss", float("nan"))))
        train_rmse.append(float(d.get("train_rmse", float("nan"))))
        val_rmse.append(float(d.get("val_rmse", float("nan"))))

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(epochs, train_loss, label="train_loss")
    ax[0].plot(epochs, val_loss, label="val_loss")
    ax[0].set_title("Loss")
    ax[0].set_xlabel("Epoch")
    ax[0].legend()

    ax[1].plot(epochs, train_rmse, label="train_rmse")
    ax[1].plot(epochs, val_rmse, label="val_rmse")
    ax[1].set_title("RMSE")
    ax[1].set_xlabel("Epoch")
    ax[1].legend()

    plt.tight_layout()
    plt.show()

    best_path = run_dir / "best.ckpt"
    last_path = run_dir / "last.ckpt"
    print("best.ckpt:", best_path if best_path.exists() else "missing")
    print("last.ckpt:", last_path if last_path.exists() else "missing")


## Next Step
Use the resulting checkpoint in the inference notebook:
- `examples/merra2_prediction/PrithviWxC_MERRA2_O3_inference.ipynb`
- set `CHECKPOINT_PATH = <base_dir>/checkpoints/<RUN_NAME>/best.ckpt`
